# Chemotherapy response prediction

Trains a 5-fold cross-validated logistic-regression ensemble on the TransNEO
chemotherapy cohort, then evaluates the held-out predictions plus IMPRESS and
PBCP. Adapted from `scr/new_tras_clusters/auc_final_chemo_v3.py`.

**Pipeline per fold:**
`MinMaxScaler → SelectKBest(f_classif, k='all') → LogisticRegression(C=100, L1)`

**Inputs**
- `../../clustering/data/transneo_chemo_cluster_props.csv` — 93 TransNEO chemo slides × 11 ST cluster proportions (training).
- `../../clustering/data/impress_chemo_cluster_props.csv` — 64 IMPRESS chemo slides (external test).
- `../../clustering/data/pbcp_chemo_cluster_props.csv` — 19 PBCP chemo slides (external test).
- `../data/response_labels.csv` — pCR / non-pCR per slide.

**Output**
- `../models/chemo_ensemble.joblib` — fitted 5-fold ensemble pipeline.

## Setup

In [1]:
import sys
sys.path.append("../lib")

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

from model_classes import EnsembleModel, FeatureAligner

import warnings; warnings.filterwarnings("ignore")

def per_patient_props(per_domain_csv):
    """Sum proportion_of_spots per (slide_name, predicted_cluster), then pivot."""
    d = pd.read_csv(per_domain_csv)
    return (d.groupby(['slide_name', 'predicted_cluster'])['proportion_of_spots']
              .sum().reset_index()
              .pivot(index='slide_name', columns='predicted_cluster', values='proportion_of_spots')
              .fillna(0))

/home/shulmaned/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Load training data — TransNEO chemo

In [2]:
labels = pd.read_csv("../data/response_labels.csv", index_col=0)
labels.index = labels.index.astype(str)

X_train = per_patient_props("../../clustering/data/transneo_chemo_per_domain.csv")
X_train.index = X_train.index.astype(str)

# Use only slides that have a Response label
keep = X_train.index.intersection(labels[labels["Response"].notna()].index)
X_train = X_train.loc[keep]
y_train = labels.loc[keep, "Response"].astype(int).values
print(f"TransNEO chemo: {X_train.shape[0]} slides, {X_train.shape[1]} features, prevalence={y_train.mean():.3f}")

TransNEO chemo: 93 slides, 10 features, prevalence=0.226


## Train — 5-fold stratified CV

In [3]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
param_grid = {
    "select__k":      ["all"],
    "logreg__penalty": ["l1"],
    "logreg__C":       [100],
    "logreg__solver":  ["saga"],
}

models, cv_scores, cv_y, cv_idx = [], [], [], []
for tr, te in kf.split(X_train, y_train):
    pipe = Pipeline([
        ("scaler", MinMaxScaler()),
        ("select", SelectKBest(f_classif)),
        ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    gs = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc",
                      refit="logreg__C").fit(X_train.iloc[tr], y_train[tr])
    models.append(gs.best_estimator_)
    proba = gs.best_estimator_.predict_proba(X_train.iloc[te])[:, 1]
    cv_scores.extend(proba); cv_y.extend(y_train[te]); cv_idx.extend(X_train.iloc[te].index)

cv_pred = pd.DataFrame({"y_true": cv_y, "y_proba": cv_scores}, index=cv_idx)
auc_train_cv = roc_auc_score(cv_pred["y_true"], cv_pred["y_proba"])
print(f"TransNEO 5-fold CV AUC: {auc_train_cv:.4f}")

TransNEO 5-fold CV AUC: 0.7530


## Build the final ensemble + save

In [4]:
ensemble_pipeline = Pipeline([
    ("aligner", FeatureAligner(expected_features=list(X_train.columns),
                                fill_values=X_train.mean().to_dict())),
    ("ensemble", EnsembleModel(models=models)),
])
joblib.dump(ensemble_pipeline, "../models/chemo_ensemble.joblib")
print("Saved ../models/chemo_ensemble.joblib")

Saved ../models/chemo_ensemble.joblib


## Evaluate on the external cohorts

IMPRESS and PBCP — load per-patient cluster proportions, intersect with
Response labels, score with the saved ensemble pipeline.

In [5]:
def evaluate(per_domain_csv, cohort_name):
    Xc = per_patient_props(per_domain_csv)
    Xc.index = Xc.index.astype(str)
    common = Xc.index.intersection(labels[labels["Response"].notna()].index)
    if len(common) == 0:
        return pd.DataFrame()
    Xc = Xc.loc[common]
    yc = labels.loc[common, "Response"].astype(int).values
    proba = ensemble_pipeline.predict_proba(Xc)
    auc = roc_auc_score(yc, proba) if len(np.unique(yc)) > 1 else float("nan")
    print(f"{cohort_name:<10} n={len(yc):>3}, prevalence={yc.mean():.3f}, AUC={auc:.4f}")
    return pd.DataFrame({"y_true": yc, "y_proba": proba, "Cohort": cohort_name}, index=common)

cv_pred["Cohort"] = "TransNEO (5-fold CV)"
print(f"TransNEO   n={len(cv_pred):>3}, prevalence={cv_pred['y_true'].mean():.3f}, AUC={auc_train_cv:.4f}")
df_impress = evaluate("../../clustering/data/impress_chemo_per_domain.csv", "IMPRESS")
df_pbcp    = evaluate("../../clustering/data/pbcp_chemo_per_domain.csv",    "PBCP")

all_pred = pd.concat([cv_pred, df_impress, df_pbcp])
all_pred.head()

TransNEO   n= 93, prevalence=0.226, AUC=0.7530
IMPRESS    n= 64, prevalence=0.422, AUC=0.7492
PBCP       n= 19, prevalence=0.263, AUC=0.8857


,y_true,y_proba,Cohort
BC_00012_470662,1,0.627870,TransNEO (5-fold CV)
BC_00014_470664,0,0.627870,TransNEO (5-fold CV)
BC_00037_473337,1,0.627870,TransNEO (5-fold CV)
BC_00038_473338,0,0.627870,TransNEO (5-fold CV)
BC_00039_473339,0,0.275404,TransNEO (5-fold CV)


## AUC summary

In [6]:
rows = []
for c, g in all_pred.groupby("Cohort"):
    rows.append({"Cohort": c, "n": len(g), "Prevalence": g["y_true"].mean(),
                  "AUC": roc_auc_score(g["y_true"], g["y_proba"]) if len(np.unique(g["y_true"])) > 1 else float("nan")})
summary = pd.DataFrame(rows).sort_values("Cohort").reset_index(drop=True)
summary.round(3)

,Cohort,n,Prevalence,AUC
0,IMPRESS,64,0.422,0.749
1,PBCP,19,0.263,0.886
2,TransNEO (5-fold CV),93,0.226,0.753
